# S10 – Unidad 1. Evidencia de Aprendizaje 1
## Carga y creación de dataset

**Integrantes:** Daniela Cataño Hernández · John Freddy Ocampo Ochoa · Óscar Alonso De La Pava Suárez

**Flujo:** `HFDatasetLoader` → DataFrame → transformaciones → `DatasetSummarizer` → `resumen_dataset.txt`

Las clases contienen la lógica de carga y de resumen; este notebook coordina el flujo y realiza las transformaciones.

## 1. Importar librerías y clases

In [1]:
import pandas as pd
from dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

print("pandas", pd.__version__)

pandas 3.0.6


## 2. Cargar el dataset desde Hugging Face
Se usa `scikit-learn/iris` (150 flores, 4 medidas y la especie).

In [2]:
loader = HFDatasetLoader(dataset_id="scikit-learn/iris", split="train")
df_original = loader.load_data()
print("Dimensiones originales:", df_original.shape)
print("Columnas originales:", list(df_original.columns))
df_original.head()

Dimensiones originales: (150, 6)
Columnas originales: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 3. Transformación 1 – Renombrar columnas


In [3]:
nombres_nuevos = {
    "Id": "id",
    "SepalLengthCm": "largo_sepalo_cm",
    "SepalWidthCm": "ancho_sepalo_cm",
    "PetalLengthCm": "largo_petalo_cm",
    "PetalWidthCm": "ancho_petalo_cm",
    "Species": "especie",
}
df_t1 = df_original.rename(columns=nombres_nuevos)

print("Antes:  ", list(df_original.columns))
print("Después:", list(df_t1.columns))
df_t1.head()

Antes:   ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']
Después: ['id', 'largo_sepalo_cm', 'ancho_sepalo_cm', 'largo_petalo_cm', 'ancho_petalo_cm', 'especie']


,id,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 4. Transformación 2 – Seleccionar columnas


In [4]:
columnas_trabajo = ["largo_sepalo_cm", "ancho_sepalo_cm",
                    "largo_petalo_cm", "ancho_petalo_cm", "especie"]
df_t2 = df_t1[columnas_trabajo].copy()

print("Columnas antes:", df_t1.shape[1], "→ después:", df_t2.shape[1])
df_t2.head()

Columnas antes: 6 → después: 5


,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


## 5. Transformación 3 – Limpiar texto y crear una columna calculada


In [5]:
df_t3 = df_t2.copy()
df_t3["especie"] = df_t3["especie"].str.replace("Iris-", "", regex=False)
df_t3["area_petalo_cm2"] = (df_t3["largo_petalo_cm"] * df_t3["ancho_petalo_cm"]).round(2)

print("Especies antes:  ", sorted(df_t2["especie"].unique()))
print("Especies después:", sorted(df_t3["especie"].unique()))
df_t3.head()

Especies antes:   ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
Especies después: ['setosa', 'versicolor', 'virginica']


,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,area_petalo_cm2
0,5.1,3.5,1.4,0.2,setosa,0.28
1,4.9,3.0,1.4,0.2,setosa,0.28
2,4.7,3.2,1.3,0.2,setosa,0.26
3,4.6,3.1,1.5,0.2,setosa,0.30
4,5.0,3.6,1.4,0.2,setosa,0.28


## 6. Transformación 4 – Filtrar filas


In [6]:
umbral = 5.0
df_final = df_t3[df_t3["largo_sepalo_cm"] >= umbral].reset_index(drop=True)

print(f"Filas antes del filtro: {len(df_t3)}")
print(f"Filas después del filtro (largo_sepalo_cm >= {umbral}): {len(df_final)}")
print(df_final["especie"].value_counts())
df_final.head()

Filas antes del filtro: 150
Filas después del filtro (largo_sepalo_cm >= 5.0): 128
especie
versicolor    49
virginica     49
setosa        30
Name: count, dtype: int64


,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,area_petalo_cm2
0,5.1,3.5,1.4,0.2,setosa,0.28
1,5.0,3.6,1.4,0.2,setosa,0.28
2,5.4,3.9,1.7,0.4,setosa,0.68
3,5.0,3.4,1.5,0.2,setosa,0.30
4,5.4,3.7,1.5,0.2,setosa,0.30


## 7. Generar el resumen TXT del DataFrame transformado


In [7]:
resumidor = DatasetSummarizer(df_final)
archivo_resumen = resumidor.generate_summary("resumen_dataset.txt")

with open(archivo_resumen, encoding="utf-8") as archivo:
    print(archivo.read())

RESUMEN DEL DATASET TRANSFORMADO
Generado: 2026-10-25 07:02

DIMENSIONES
-----------
Cantidad de filas: 128
Cantidad de columnas: 6

NOMBRES DE LAS COLUMNAS
-----------------------
largo_sepalo_cm, ancho_sepalo_cm, largo_petalo_cm, ancho_petalo_cm, especie, area_petalo_cm2

TIPOS DE DATOS
--------------
largo_sepalo_cm    float64
ancho_sepalo_cm    float64
largo_petalo_cm    float64
ancho_petalo_cm    float64
especie                str
area_petalo_cm2    float64

VALORES NULOS POR COLUMNA
-------------------------
largo_sepalo_cm    0
ancho_sepalo_cm    0
largo_petalo_cm    0
ancho_petalo_cm    0
especie            0
area_petalo_cm2    0
Total de valores nulos: 0

ESTADISTICAS DESCRIPTIVAS (columnas numéricas)
----------------------------------------------
                 count   mean    std   min   25%   50%     75%    max
largo_sepalo_cm  128.0  6.041  0.727  5.00  5.50  6.00   6.500   7.90
ancho_sepalo_cm  128.0  3.054  0.452  2.00  2.80  3.00   3.325   4.40
largo_petalo_cm  128.0 

## 8. Comprobación final

In [8]:
print(f"DataFrame original: {df_original.shape[0]} filas × {df_original.shape[1]} columnas")
print(f"DataFrame final:    {df_final.shape[0]} filas × {df_final.shape[1]} columnas")
print("Resumen guardado en:", archivo_resumen)

DataFrame original: 150 filas × 6 columnas
DataFrame final:    128 filas × 6 columnas
Resumen guardado en: resumen_dataset.txt


## Explicación de las transformaciones

| # | Transformación | Operación | Efecto |
|---|---|---|---|
| 1 | Renombrar columnas | `rename` | Nombres en español y en minúsculas |
| 2 | Seleccionar columnas | `df[[...]]` | Se elimina `id` (6 → 5 columnas) |
| 3 | Limpiar texto y crear columna | `str.replace` + operación aritmética | Especie sin prefijo `Iris-`; nueva columna `area_petalo_cm2` (5 → 6 columnas) |
| 4 | Filtrar filas | máscara booleana | Solo flores con sépalo ≥ 5 cm (150 → menos filas) |

El archivo `resumen_dataset.txt` corresponde al DataFrame final, después de las cuatro transformaciones.